In [2]:
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util as qz
import ifcopenshell.util.element as qx
model = ifcopenshell.open("test.ifc")

# model = ifcopenshell.open("AC20-FZK-Haus.ifc")


# declare wall objects IfcWall
walls = model.by_type('IfcWall')
print(walls)


# wall = model.by_type('IfcWall')[0]
# print(wall.id())


# def get_opening_list_from_wall(wall):
#     """
#     Get the list of openings associated with the given wall.
 
#     :param wall: The wall object.
#     :type wall: IfcWall
 
#     :return: List of IfcOpening objects associated with the wall.
#     :rtype: list
#     """
#     opening_list = wall.HasOpenings
#     return opening_list

[#18=IfcWall('3dilnCT4X0FeORtEHkzAl6',$,$,$,$,#49,#51,$,$), #19=IfcWall('2Z4XZBDA1Dyv8Wx5pwm0D_',$,$,$,$,#44,#50,$,$), #52=IfcWall('1kBznzoxnC7RIFJm4WN2Ga',$,$,$,$,#58,#64,$,$), #53=IfcWall('1y6irwaUv7Cudt4Ur5fbSR',$,$,$,$,#63,#65,$,$), #104=IfcWall('0RjvbqWNrBkxP0XWT6j0JG',$,$,$,$,#130,#137,$,$), #105=IfcWall('1rHkP_6XD0q9wggFi$ildY',$,$,$,$,#135,#136,$,$), #138=IfcWall('2QGKSLCbH7yQhAW5inX__Y',$,$,$,$,#143,#144,$,$), #183=IfcWall('3u8G0nOa91SxsN5XrsBunu',$,$,$,$,#208,#209,$,$), #210=IfcWall('3tM71vypX0MQdr7tlABLLV',$,$,$,$,#221,#222,$,$), #211=IfcWall('3i2ghnFjH51urOXWzZSP0w',$,$,$,$,#216,#223,$,$)]


In [3]:
# number of walls. For this model, "walls" only has 4 elements, which makes it easy
no_walls = len(walls)
print('Number of walls is ', no_walls)

print('Properties are ', walls[0])

wall = walls[0]

print('Loop start--------')
for i in range(0,len(wall)):
    print('Type is ',type(wall[i]),'. Value is ', wall[i])


# this is just a way to get direct property sets from the Ifcwall. Likely need to look at inverse relationship formulas
# ifcopenshell.entity_instance.entity_instance

Number of walls is  10
Properties are  #18=IfcWall('3dilnCT4X0FeORtEHkzAl6',$,$,$,$,#49,#51,$,$)
Loop start--------
Type is  <class 'str'> . Value is  3dilnCT4X0FeORtEHkzAl6
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #49=IfcLocalPlacement($,#48)
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #51=IfcProductDefinitionShape($,$,(#29))
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'NoneType'> . Value is  None


In [4]:
# check which walls have openings
for i in range(len(walls)):
    wallOpen = walls[i].HasOpenings
    if (wallOpen):
        print('Wall ',i,' has ', len(wallOpen),' openings')

Wall  0  has  1  openings
Wall  4  has  1  openings
Wall  8  has  1  openings


In [5]:
# example obtaining opening dimensions from a single wall
for rel in walls[0].HasOpenings:
    opening = rel.RelatedOpeningElement
    print("Name is ", opening.Name)
    
    # this get property sets for the specific opening
    psets = qx.get_psets(opening)

    # This determines the exact name of the "proprties" used. Sometimes it could be pset, or qto when finding window dimensions
    # print(psets.keys())

    # since the dimensions could be stored in two different properties, we declare variables that can compute both i.e. one will be legit, the other will be empty
    qto = psets.get("Qto_OpeningElementBaseQuantities",{})
    pset_common = psets.get('Pset_OpeningElementCommon',{})

    width = qto.get("Width") or pset_common.get("Width")
    height = qto.get("Height") or pset_common.get("Height")
    depth = qto.get("Depth") or pset_common.get("Depth")
    print("WxHxD = ",width,'x',height,'x',depth, '\n')

Name is  None
WxHxD =  None x None x None 



For my next trick, I'm going to try to obtain the wall property materials

In [6]:
for wall in walls:
    material = qx.get_material(wall)

    # first, work out the number of layers, and the materials used
    if (material.is_a("IfcMaterialLayerSetUsage")):
        layer_set = material.ForLayerSet
        print('Number of Wall Layers is ',len(layer_set.MaterialLayers))

        for layer in layer_set.MaterialLayers:
            # obtain material name
            mat_name = layer.Material.Name
            thickness = layer.LayerThickness
            print('Material is ',mat_name,'\n Thickness is ',thickness,'\n')


            # obtain thermal properties of the wall layer
            # https://standards.buildingsmart.org/IFC/RELEASE/IFC2x3/TC1/HTML/ifcmaterialpropertyresource/lexical/ifcthermalmaterialproperties.htm
            thermal_pset = qx.get_psets(layer.Material).get("Pset_MaterialThermal",{})
            conductivity = thermal_pset.get("ThermalConductivity")
            specific_heat = thermal_pset.get("SpecificHeatCapacity")
            # print(specific_heat)

AttributeError: 'NoneType' object has no attribute 'is_a'

In [ ]:
for wall in walls:
    material = qx.get_material(wall)

Aim:
To extract information from Ifc models using IfcOpenShell for use in designing the Fire Safety strategy

Objectives:
- Obtain property materials from a room, and use it to develop a design fire
- Obtain compartment size dimensions and opening, to calculate flashover conditions during a compartment fire
- Automatically compute travel distances within a building

OR...

Aim:
To evaluate travel distances within a building from Ifc models using IfcOpenShell

Objectives:
- Construct rooms within a building using nodes and graphs
- Compute travel distance between nodes using shortest straight line path
- Find longest paths of travel (this looks hard)
- Validate travel distance using pathfinder

https://en.wikipedia.org/wiki/Price%27s_model


In [ ]:
# Extract IFC Spaces
spaces = model.by_type('IfcSpace')
print(f"Number of spaces: {len(spaces)}")

for space in spaces:
    print(f"Space ID: {space.id()}, Name: {space.Name}")
    
    # Get geometry and position
    try:
        # Create geometry settings
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)
        
        # Get the shape
        shape = ifcopenshell.geom.create_shape(settings, space)
        
        # Get vertices (geometry data)
        verts = shape.geometry.verts
        print(f"Space {space.Name} has {len(verts)//3} vertices")
        
        # For position, you can get the placement
        placement = space.ObjectPlacement
        if placement:
            if placement.is_a('IfcLocalPlacement'):
                # Get the relative placement
                rel_placement = placement.RelativePlacement
                if rel_placement and rel_placement.is_a('IfcAxis2Placement3D'):
                    location = rel_placement.Location
                    print(f"Position: ({location.Coordinates[0]}, {location.Coordinates[1]}, {location.Coordinates[2]})")
                    
    except Exception as e:
        print(f"Error getting geometry for space {space.Name}: {e}")

Number of spaces: 0
